In [5]:
from datetime import timedelta, datetime
import time
import requests
from typing import Dict, Any, Union
import json
import os

import logging
logger = logging.getLogger(__name__)

In [6]:
PROJECT_NAME = "nyc-transport-pipeline"

API_CONFIG = {
    'traffic_speed': {
        'base_url': 'https://data.cityofnewyork.us/resource/i4gi-tjb9.json',
        'limit': 50000,
        'order_by': 'data_as_of ASC',
        'date_field': 'data_as_of',
    },
    '311_requests': {
        'base_url': 'https://data.cityofnewyork.us/resource/erm2-nwe9.json',
        'limit': 50000,
        'order_by': 'created_date ASC',
        'date_field': 'created_date',
    },
    'traffic_volume': {
        'base_url': 'https://data.cityofnewyork.us/resource/btm5-ppia.csv',
        'limit': 50000,
        'date_field': 'yr',  # Utilise yr, m, d séparément
    },
    'weather': {
        'base_url': 'https://www.ncei.noaa.gov/access/services/data/v1',
        'params': {
            'dataset': 'daily-summaries',
            'dataTypes': 'PRCP,TMAX,TMIN,SNOW',
            'stations': 'USW00094728',
            'format': 'json',
            'units': 'metric',
        },
    },
}



PROJECT_DATA_COLLECTOR_NAME = "NYC_TRANSPORTATION_DATA_COLLECT"


def get_collected_tags(collected_name:str, frequency:str="days"):
    
    all_collected_tags = {
        "hour": [
            PROJECT_DATA_COLLECTOR_NAME, 
            "frequency:hourly",
            "schedule:every-hour",
            "offset-1",
            "retention:7d",
            "priority:normal"
        ],
        "days": [
            PROJECT_DATA_COLLECTOR_NAME, 
            "frequency:daily",
            "schedule:daily",
            "offset-1",
            "retention:90d",
            "priority:high"
        ],
        "weekly": [
            PROJECT_DATA_COLLECTOR_NAME, 
            "frequency:weekly",
            "schedule:weekly",
            "offset-1",
            "retention:1y",
            "priority:low"
        ],
        "monthly": [
            PROJECT_DATA_COLLECTOR_NAME, 
            "frequency:monthly",
            "schedule:monthly",
            "offset-1",
            "retention:3y",
            "priority:high"
        ],
        "yearly": [
            PROJECT_DATA_COLLECTOR_NAME, 
            "frequency:yearly",
            "schedule:yearly",
            "offset-1",
            "retention:10y",
            "priority:low"
        ],
    }

    if frequency not in all_collected_tags.keys():
        raise ValueError(
                    f"Fréquence '{frequency}' invalide. "
                    f"Valeurs acceptées: {list(all_collected_tags.keys())}"
                )    
    return all_collected_tags[frequency]+[collected_name]



def fetch_data_from_url(url:str,timeout: int = 30)-> Dict[str, Any]:
    try:
        # Effectuer la requête GET
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        
        # Récupérer les informations de base
        content_type = response.headers.get('Content-Type', '').lower()
        encoding = response.encoding or 'utf-8'
        
        # Déterminer le format et parser si possible
        result = {
            'status_code': response.status_code,
            'content_type': content_type,
            'encoding': encoding,
            'headers': dict(response.headers),
            'url': response.url,
            'data': None,
            'format': 'unknown',
            'raw_content': response.content
        }
        
        # Détecter et parser selon le format
        if 'application/json' in content_type or url.endswith('.json'):
            try:
                result['data'] = response.json()
                result['format'] = 'json'
            except json.JSONDecodeError:
                result['data'] = response.text
                result['format'] = 'text'
        
        elif 'text/csv' in content_type or url.endswith('.csv'):
            result['data'] = response.text
            result['format'] = 'csv'
        
        elif 'application/xml' in content_type or 'text/xml' in content_type or url.endswith('.xml'):
            result['data'] = response.text
            result['format'] = 'xml'
        
        elif 'text/html' in content_type:
            result['data'] = response.text
            result['format'] = 'html'
        
        elif 'text/' in content_type:
            result['data'] = response.text
            result['format'] = 'text'
        
        else:
            # Format binaire (images, PDFs, etc.)
            result['data'] = response.content
            result['format'] = 'binary'
        return result
        
    except requests.exceptions.Timeout:
        raise requests.RequestException(f"Timeout: La requête a dépassé {timeout} secondes")
    except requests.exceptions.RequestException as e:
        raise requests.RequestException(f"Erreur lors de la requête: {str(e)}")

In [39]:
COLLECTED_NAME = "Comptages_véhicules_intersection"
API_LABEL = "traffic_volume"

In [79]:
from urllib.parse import urlencode
import csv
from io import StringIO

logger.info("DÉBUT EXTRACTION TRAFFIC SPEED")
execution_date = datetime(year=2011,month=1,day=9)
config = API_CONFIG[API_LABEL]
base_url = config['base_url']
limit = config['limit']

date={'yr': execution_date.year,'m': execution_date.month,'d': execution_date.day}
yr = date.get('yr')
m = date.get('m')
d = date.get('d')

In [ ]:
url = (
        f"{base_url}?"
        f"$limit={limit}"
        f"&$offset={0}"
        f"&$where=date='{yr:04d}-{m:02d}-{d:02d}'"
    )

print(url)
results = fetch_data_from_url(url)
print(results['format'] == 'csv')

https://data.cityofnewyork.us/resource/btm5-ppia.csv?$limit=50000&$offset=0&$where=date='2012-01-09'
True


In [68]:
reader = csv.reader(StringIO(results['data']))
data = list(reader)
len(data)

103

In [64]:
len(data)

103

In [66]:
header = data[0]
len(header)

31

In [67]:
csv_data = data[1:]
len(csv_data)

102

In [80]:
all_data = []
max_retries=3 #Nombre de tentative
page=1 # Init page
offset=0
limit=10
while True:
    nb_lignes = 0
    for attempt in range(max_retries):
        url = (
            f"{base_url}?"
            f"$limit={limit}"
            f"&$offset={offset}"
            f"&$where=date='{yr:04d}-{m:02d}-{d:02d}'"
        )
        try:
            results = fetch_data_from_url(url)
            logger.info(f"format de la réponse {results['format']}")

            if results['format'] != 'csv':
                raise Exception(f"Format incorrecte, nous attendons un fichier csv.") 
            
            reader = csv.reader(StringIO(results['data']))
            data = list(reader)


            header = data[0]
            content = data[1:]
            nb_lignes = len(content)

            if page == 1 : # #si premiere page, on récupére le header des données
                logger.info(f"Premiere page, longueur du header {len(header)}")
                all_data.append(header)

            all_data.extend(content)
            offset += limit
            page +=1

            break


        except requests.exceptions.Timeout:
            logger.warning(f"Timeout (tentative {attempt + 1}/{max_retries})")
            if attempt == max_retries - 1:
                raise Exception(f"Timeout après {max_retries} tentatives")
            time.sleep(2 ** attempt)  # Backoff exponentiel
            
        except requests.exceptions.RequestException as e:
            logger.error(f"Erreur API: {e}")
            if attempt == max_retries - 1:
                raise Exception(f"Erreur API après {max_retries} tentatives: {e}")
            time.sleep(2 ** attempt)

    if nb_lignes < limit:
        logger.info(f"Fin de pagination: dernière page avec {nb_lignes} lignes")
        break

In [75]:
len(all_data)

103

In [ ]:
import pandas as pd
df =pd.DataFrame(all_data, )
df

,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,id,segmentid,roadway_name,from,to,direction,date,_12_00_1_00_am,_1_00_2_00am,_2_00_3_00am,...,_2_00_3_00pm,_3_00_4_00pm,_4_00_5_00pm,_5_00_6_00pm,_6_00_7_00pm,_7_00_8_00pm,_8_00_9_00pm,_9_00_10_00pm,_10_00_11_00pm,_11_00_12_00am


In [ ]:
def extract_data(ti,**kwargs):
    logger.info("DÉBUT EXTRACTION TRAFFIC SPEED")

    execution_date = kwargs.get('logical_date') or kwargs.get('execution_date')

    config = API_CONFIG[API_LABEL]

    base_url = config['base_url']
    limit = config['limit']

    date=build_datetime_range(execution_date)
    yr = date.get('yr')
    m = date.get('m')
    d = date.get('d')

    # Informations de debug
    logger.info(f"Configuration:")
    logger.info(f"Source: {COLLECTED_NAME}")
    logger.info(f"URL: {url}")
    logger.info(f"Limit: {limit}")
    logger.info(f"Execution date: {execution_date}")

    all_data = []
    max_retries=3 #Nombre de tentative
    page=1 # Init page
    offset=0
    while True:
        nb_lignes = 0
        for attempt in range(max_retries):
            url = (
                f"{base_url}?"
                f"$limit={limit}"
                f"&$offset={offset}"
                f"&$where=date='{yr:04d}-{m:02d}-{d:02d}'"
            )
            try:
                results = fetch_data_from_url(url)
                logger.info(f"format de la réponse {results['format']}")

                if results['format'] != 'csv':
                    raise Exception(f"Format incorrecte, nous attendons un fichier csv.") 
                
                reader = csv.reader(StringIO(results['data']))
                data = list(reader)


                header = data[0]
                content = data[1:]
                nb_lignes = len(content)

                if page == 1 : # #si premiere page, on récupére le header des données
                    logger.info(f"Premiere page, longueur du header {len(header)}")
                    all_data.append(header)

                all_data.extend(content)
                offset += limit
                page +=1

                break


            except requests.exceptions.Timeout:
                logger.warning(f"Timeout (tentative {attempt + 1}/{max_retries})")
                if attempt == max_retries - 1:
                    raise Exception(f"Timeout après {max_retries} tentatives")
                time.sleep(2 ** attempt)  # Backoff exponentiel
                
            except requests.exceptions.RequestException as e:
                logger.error(f"Erreur API: {e}")
                if attempt == max_retries - 1:
                    raise Exception(f"Erreur API après {max_retries} tentatives: {e}")
                time.sleep(2 ** attempt)

        if nb_lignes < limit:
            logger.info(f"Fin de pagination: dernière page avec {nb_lignes} lignes")
            break     

    logger.info(f"longueur des données: {len(all_data)}")
        
    # Validation
    if not all_data:
        raise ValueError(
            f"Aucune donnée extraite à la date du {date}"
        )
        
    nb_records = len(all_data[1:])

    logger.info(f"Extraction réussie: {nb_records} enregistrements")
        
    # Stocker dans XCom pour la tâche suivante
    ti.xcom_push(key='traffic_volume', value=all_data)
    ti.xcom_push(key='nb_records', value=nb_records)
    ti.xcom_push(key='date', value=date)
                
    return nb_records